## The State of Tax Justice: Data cleaning

- Author: Alison Schultz, based on Javier Garcia-Bernado's work
- Created: 4 August 2023
- Last updated: 11 August 2024

**Description**
- This notebook is one out of three notebooks to estimate the tax losses caused by profit shifting by multinational enterprises (MNEs). The analysis used the misalignment method based on the country-by-country reports (CbCR) published by the OECD.
- This notebook imports and cleans all data. It produces the dataset "data/intermediate/imputation_sample.csv" that is used to impute missing values in the second notebook and estimate misalignment in the third notebook.

**Outline**
1. Import and clean CbCR data
    1.1 Import CbcR data
    1.2 Correct for dividend double counting
    1.3 Calculate ETRs
2. Import and clean other data needed for the analysis
3. Prepare dataset for imputation


**To dos before running this notebook**
1. Adjust years for which the analysis should be run in 'sotj_profit_shifting_estimates/config.py'
2. Store the most up-to-date versions of the following datasets in 'sotj_profit_shifting_estimates/data/input' and adjust the file names in 'sotj_profit_shifting_estimates/config.py' if necessary. 
    - CbCR data (Table 1) for all years you would like to analyze from the OECD: https://stats.oecd.org/Index.aspx?DataSetCode=CBCR_TABLEI
    - Corporate income tax rate data from the Tax Foundation (panel version): https://taxfoundation.org/data/all/global/corporate-tax-rates-by-country-2022/
    - Wage data from the ILO: https://www.ilo.org/ilostat-files/WEB_bulk_download/indicator/EAR_4MTH_SEX_ECO_CUR_NB_A.csv.gz
    - GDP and population data from the 
    - Public health expenditure data from the WHO: 
3. Check the following resources regarding the CbCR data and adjust Section 1.2 of this notebook accordingly.
    - Most recent version of OECD disclaimer: https://www.oecd.org/tax/tax-policy/anonymised-and-aggregated-cbcr-statistics-disclaimer.pdf
    - Country notes (instert country of interest for COUNTRY): https://www.oecd.org/tax/tax-policy/COUNTRY-cbcr-country-specific-analysis.pdf
    - Skim through raw data for potential misreporting (e.g. very high values for ncertain countries where no such values are expected)
4. Load the environment "sotj_profit_shifting_estimates" (available via 'sotj_profit_shifting_estimates/environment.yml')


In [1]:
import pandas as pd
import numpy as np
from config import *
import tjn_tools
import statsmodels.formula.api as smf
from itertools import permutations, product

### 1. Import and clean CbCR data
- 1.1 Import CbCR data
- 1.2 Clean CbCR data
- 1.3 Calculate ETRs

#### 1.1 Import CbCR data

In [ ]:
cbcr_variables = ['REF_AREA', 'Reference area', 'COUNTERPART_AREA', 'Counterpart area','Profit grouped by', 'TIME_PERIOD', 'Unrelated party revenues',
                  'Profit (loss) before income tax', 'Adjusted profit (loss) before income tax', 'Income tax paid (on cash basis)', 
                  'Income tax accrued - current year','Employees', 'Tangible assets other than cash and cash equivalents', 
                  'Stated capital', 'Total revenues', 'Related party revenues', 'Holding or managing intellectual property business activity',
                  'Multinational enterprise groups','Multinational enterprise sub-groups','Entities']
cbcr_long = pd.read_csv(cbcr_data)
cbcr_wide = pd.pivot_table(cbcr_long, index=['REF_AREA', 'Reference area', 'COUNTERPART_AREA', 'Counterpart area','Profit grouped by', 'TIME_PERIOD'],
                           values="OBS_VALUE", columns="Measure").reset_index()
cbcr = cbcr_wide[cbcr_variables]

# Remove non-existing jurisdictions and stateless income
cbcr = cbcr[(cbcr['COUNTERPART_AREA'] != 'ANT_F') & (cbcr['COUNTERPART_AREA'] != 'BVT') & (cbcr['COUNTERPART_AREA'] != 'STLS')] # delete rows for Netherland Antilles as they do not exist anymore and are only incorrectly reported by Mexico, and Stateless entities

# Rename CBCR variables 
cbcr.rename(columns = {
    'REF_AREA':'iso_parent',
    'Reference area':'parent_jurisdiction',
    'COUNTERPART_AREA':'iso_partner',
    'Counterpart area':'partner_jurisdiction',
    'Profit grouped by':'grouping',
    'TIME_PERIOD':'year',
    'Unrelated party revenues': 'unrelated_party_revenues',
    'Profit (loss) before income tax':'profit_loss_before_income_tax',
    'Adjusted profit (loss) before income tax':'adjusted_profit_loss_before_income_tax',
    'Income tax paid (on cash basis)':'income_tax_paid_on_cash_basis',
    'Income tax accrued - current year':'income_tax_accrued_current_year',
    'Employees':'n_employees',
    'Tangible assets other than cash and cash equivalents':'tangible_assets_except_cash',
    'Stated capital': 'stated_capital',
    'Total revenues': 'total_revenues',
    'Related party revenues':'related_party_revenues',
    'Holding or managing intellectual property business activity': 'holding_or_managing_ip',
    'Multinational enterprise groups':'n_cbcr',
    'Multinational enterprise sub-groups':'n_cbcr_groups',
    'Entities':'n_entities'
    },
    inplace = True)

In [ ]:
# Sample of reporting jurisdictions and partner jurisdictions
parent_countries = cbcr['iso_parent'].drop_duplicates()
partner_countries  = cbcr['iso_partner'].drop_duplicates()

#### 1.2 Clean CBCR data: Adjust for the double counting of dividends
The CbCR data has the problem that dividends are partly double counted. We correct for this double counting, according to the following sources and considerations. The correction is applied to all subgroups (if their total profits are > 0) and the subgroup with positive profits. TO DO: REMOVE FTJ
- Argentina-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Australia-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Austria-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Belgium based MNCs
    - 2018:
        - domestic: We reduce 50% of profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Bermuda based MNCs
    - 2018:
        - domestic: We reduce 50% of profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Brazil-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Canada-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Switzerland-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Chile-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Cayman Islands based MNCs
    - 2018: MNCs have negative profits, so double counting is unlikely, no reduction in domestic or foreign profits
- Denmark-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- France-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Germany-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Greece-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Hong-Kong-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Indonesia-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- India-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Ireland-based MNCs: CBCR Country notes: https://www.oecd.org/tax/tax-policy/ireland-cbcr-country-specific-analysis.pdf
    - 2016: No issues found -> no correction
    - 2018: Ireland-based MNCs have negative profits -> double counting should not be an issue, so we don't control for it
- Isle of Man based MNCs
    - 2018:
        - domestic: We reduce 50% of profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Italy-based MNCs: CBCR country notes: https://www.oecd.org/tax/tax-policy/italy-cbcr-country-specific-analysis.pdf
    - 2016: The median value of dividends is XXX, the  average value of the share of dividends is equal to 38.2%, thus implying that dividends are concentrated in few firms.
    - 2018: The mean and the median percentage of intracompany dividends estimated to be included in the CBCR, profit(loss) figure at the subgroup level is respectively 50% and 24% -> reduce profits by 50% to be conservative
        - domestic: We reduce 50% of profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Japan-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Latvia-based MNCs
    - 2018:
        - domestic: We do not reduce profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Lithuania-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Luxembourg-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Malaysia-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Mauritius-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Mexico-based MNCs:
    - 2018: According to the CBCR data, domestic ETRs similar to foreign ETRs -> likely no large double counting
        - domestic: We do not reduce profits (as domestic ETR similar to foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Netherlands based MNCs: https://www.oecd.org/tax/tax-policy/united-kingdom-cbcr-country-specific-analysis.pdf
    - 2016: 5794 out of 36802 is double counted
        - domestic: We reduce profits by 15.74%
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
    - 2017: NLD reports corrected data already in source
        - domestic: We use corrected numbers
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
    - 2018: NLD reports corrected data already in source
        - domestic: We use corrected numbers
- New Zealand-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
  Norway-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Panama-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Peru-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Portugal-based MNCs
    - 2019:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Romania-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Saudi-Arabia-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Singapore based MNCs
    - 2018:
        - domestic: We reduce 50% of profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Slovenia-based MNCs:
    - 2018: 
        - domestic: We do not reduce profits (as domestic ETR similar to foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Spain-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- South Africa-based MNCs
    - 2018:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Sweden-based MNCs: CBCR country notes: https://www.oecd.org/tax/tax-policy/sweden-cbcr-country-specific-analysis.pdf
    - 2016: Dividends share 51.95%
        - domestic: We reduce 51.59% of profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
    - 2018: "...under the assumption that all companies included dividends in their CbCR figures one can subtract USD 29.8 billion from USD 49.1 billion..."
        - domestic: We reduce 60.69% of reported profits
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- Turkey-based MNCs
    - 2019:
        - domestic: We reduce 35% of profits (as domestic ETR << than foreign ETR)
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
- United Kingdom based MNCs: https://www.oecd.org/tax/tax-policy/united-kingdom-cbcr-country-specific-analysis.pdf
    - 2016: The total value of dividends extracted for groups we believe had included intragroup dividends received was approximately £55 billion: 78.169865 out of 152.918884 = 51.1%
        - domestic: we reduce profits by 51.1%
        - foreign: We do not reduce profits (except for tax havens and groups, see below)
    - 2017: UK reports corrected data already in source -> We use corrected numbers
    - 2018: UK reports corrected data already in source -> We use corrected numbers
- US-based MNCs: García-Bernardo, Janský & Zucman (2022), https://gabriel-zucman.eu/files/GBJZ2021.pdf
    - 2016
    - 2017: domestic: 55%, foreign: 7%
    - 2018: domestic: 75%, foreign: 39%
        - Domestic profits: reduce 74% of profits
        - Foreign profits: reduce 45% of profits for tax havens (as obvious from data, not planned)
- Foreign income in tax havens and country groups
        - booked in tax havens: 
            - 2018: For US-based MNCs, García-Bernardo, Janský & Zucman (2022) find a double counting of 9% -> We remove 10% of profits booked in tax havens
        - booked in grouped jurisdictions: As some of the grouped jurisdictions are tax havens, we apply 10% for 50% of the jurisdictions -> We remove 5% of profits

We do not correct for participation results (de-mergers, takeovers and disposal), as they typically involve tax avoidance strategies in tax havens.

#### 1.2.1 Create dictionaries with all fractions by which profits need to be multiplied to correct

In [ ]:
# Add the correction values for each specific year below. The countries with adjusted data will be taken care of later.
# The cleaning is done separately for:
#...correction for domestic profits (correction per MNC home country). Please include additional years

correction_domestic_2016 = {"ARG": 0.35, "AUS": 0.35, "AUT": 0.35, "BEL": 0.5, "BMU": 0.5, "BRA": 0.35, "CAN": 0.35, "CHE": 0.35,
                            "CHL": 0.35, "CYM": 0, "CZE": 0.35, "DEU": 0.35, "DNK": 0.35, "ESP": 0.35, "FIN": 0.35, "FRA": 0.35, 
                            "GBR": 0.511, "GRC": 0.35, "HKG": 0.35, "HUN": 0.35, "IDN": 0.35, "IMN": 0.5, "IND": 0.35, "IRL": 0,
                            "ITA": 0.5, "JPN": 0.35, "KOR": 0.35, "LTU": 0.35, "LUX": 0.35, "LVA": 0, "MEX": 0, "MYS": 0.35, 
                            "NLD": 0.1574, "NOR": 0.35, "NZL": 0.35, "PAN": 0.35, "PER": 0.35, "POL": 0.35, "ROU": 0.35, "SAU": 0.35,
                            "SGP": 0.5, "SVN": 0, "SWE": 0.5159, "USA": 0.74, "ZAF": 0.35}
correction_domestic_2017 = {"ARG": 0.35, "AUS": 0.35, "AUT": 0.35, "BEL": 0.5, "BMU": 0.5, "BRA": 0.35, "CAN": 0.35, "CHE": 0.35,
                            "CHL": 0.35, "CYM": 0, "CZE": 0.35, "DEU": 0.35, "DNK": 0.35, "ESP": 0.35, "FIN": 0.35, "FRA": 0.35, 
                            "GBR": 0, "GRC": 0.35, "HKG": 0.35, "HUN": 0.35, "IDN": 0.35, "IMN": 0.5, "IND": 0.35, "IRL": 0,
                            "ITA": 0.5, "JPN": 0.35, "KOR": 0.35, "LTU": 0.35, "LUX": 0.35, "LVA": 0, "MEX": 0, "MYS": 0.35, 
                            "NLD": 0, "NOR": 0.35, "NZL": 0.35, "PAN": 0.35, "PER": 0.35, "POL": 0.35, "ROU": 0.35, "SAU": 0.35,
                            "SGP": 0.5, "SVN": 0, "SWE": 0.61, "USA": 0.55, "ZAF": 0.35}
correction_domestic_2018 = {"ARG": 0.35, "AUS": 0.35, "AUT": 0.35, "BEL": 0.5, "BMU": 0.5, "BRA": 0.35, "CAN": 0.35, "CHE": 0.35,
                            "CHL": 0.35, "CYM": 0, "CZE": 0.35, "DEU": 0.35, "DNK": 0.35, "ESP": 0.35, "FIN": 0.35, "FRA": 0.35, 
                            "GBR": 0, "GRC": 0.35, "HKG": 0.35, "HUN": 0.35, "IDN": 0.35, "IMN": 0.5, "IND": 0.35, "IRL": 0,
                            "ITA": 0.5, "JPN": 0.35, "KOR": 0.35, "LTU": 0.35, "LUX": 0.35, "LVA": 0, "MEX": 0, "MYS": 0.35, 
                            "NLD": 0, "NOR": 0.35, "NZL": 0.35, "PAN": 0.35, "PER": 0.35, "POL": 0.35, "ROU": 0.35, "SAU": 0.35,
                            "SGP": 0.5, "SVN": 0, "SWE": 0.6069, "USA": 0.74, "ZAF": 0.35} # TO DO SUBSTITUE WITH 0.75 FOR USA
correction_domestic_2019 = {"ARG": 0.35, "AUS": 0.35, "AUT": 0.35, "BEL": 0.5, "BMU": 0.5, "BRA": 0.35, "CAN": 0.35, "CHE": 0.35,
                            "CHL": 0.35, "CYM": 0, "CZE": 0.35, "DEU": 0.35, "DNK": 0.35, "ESP": 0.35, "FIN": 0.35, "FRA": 0.35, 
                            "GBR": 0, "GRC": 0.35, "HKG": 0.35, "HUN": 0.35, "IDN": 0.35, "IMN": 0.5, "IND": 0.35, "IRL": 0,
                            "ITA": 0.5, "JPN": 0.35, "KOR": 0.35, "LTU": 0.35, "LUX": 0.35, "LVA": 0, "MAC": 0.35, "MEX": 0, "MUS": 0.35,  "MYS": 0.35, 
                            "NLD": 0, "NOR": 0.35, "NZL": 0.35, "PAN": 0.35, "PER": 0.35, "POL": 0.35, "PRT": 0.35, "ROU": 0.35, "SAU": 0.35,
                            "SGP": 0.5, "SVN": 0, "SWE": 0.6069, "TUR": 0.35, "USA": 0.74, "ZAF": 0.35} # TO DO SUBSTITUE WITH 0.75 FOR USA
# From 2020 on, there was clear guidance on how to treat intracompany dividends. Therefore, there should be 0 double counting
correction_domestic_2020 = {}
correction_domestic_2021 = {}

#...correction for foreign profits (correction per MNC home country). Please note: This correction is applied IN ADDITION to the tax haven or country group correction for 
# tax havens and country groups. For instance, when the correction is 7% and the correction for tax havens is 10%, profits will be reduced by 10% and then, in addition, by 7%,
# applied to the corrected values.
correction_foreign_2016 = {"USA": 0.07} 
correction_foreign_2017 = {"USA": 0.07} 
correction_foreign_2018 = {"USA": 0.39} # TO DO SUBSTITUTE WITH ACTUAL CORRECTION 0.39 for USA # please include additional years in the same structure
correction_foreign_2019 = {"USA": 0.39} 
correction_foreign_2020 = {} 
correction_foreign_2021 = {} 
#...correction for foreign profits booked in tax havens (correction per partner jurisdiction)
correction_taxhavens_2016 = {key: 0.10 for key in tax_havens}
correction_taxhavens_2017 = {key: 0.10 for key in tax_havens}
correction_taxhavens_2018 = {key: 0.10 for key in tax_havens} 
correction_taxhavens_2019 = {key: 0.10 for key in tax_havens} 
correction_taxhavens_2020 = {} 
correction_taxhavens_2021 = {} 
#...correction for foreign profits booked in country groups (correction per partner jurisdiction)
correction_countrygroups_2016 = {key: 0.05 for key in country_groups}
correction_countrygroups_2017 = {key: 0.05 for key in country_groups}
correction_countrygroups_2018 = {key: 0.05 for key in country_groups} 
correction_countrygroups_2019 = {key: 0.05 for key in country_groups} 
correction_countrygroups_2020 = {} 
correction_countrygroups_2021 = {} # please include additional years in the same structure

corrections = {
    "domestic": {},
    "foreign": {},
    "taxhavens": {},
    "countrygroups": {}
}

for year in range(first_year, first_year + n_years + 1):
    corrections["domestic"][year] = {key: 0 for key in parent_countries.tolist()}
    corrections["foreign"][year] = {key: 0 for key in parent_countries.tolist()}
    corrections["taxhavens"][year] = {key: 0 for key in partner_countries.tolist()}
    corrections["countrygroups"][year] = {key: 0 for key in partner_countries.tolist()}
    
    # Include the correction values for the corresponding year and category
    corrections["domestic"][year].update(globals().get(f"correction_domestic_{year}", {}))
    corrections["foreign"][year].update(globals().get(f"correction_foreign_{year}", {}))
    corrections["taxhavens"][year].update(globals().get(f"correction_taxhavens_{year}", {}))
    corrections["countrygroups"][year].update(globals().get(f"correction_countrygroups_{year}", {}))

#### 1.2.2 Apply correction to main dataset

In [ ]:
def correct_for_dividend_double_counting(row, year):
    """This function corrects reported profits for dividend double counting in the CBCR dataset, storing the corrected profits in the new column "profit_loss_before_income_tax_corrected".
    Before applying it, correction details have to be defined in the dictionary corrections for domestic profits, foreign profits, profits in tax havens, and profits in country groups"""

    profit = row["profit_loss_before_income_tax"]

    # 1. Use adjusted values when available
    if pd.notna(row["adjusted_profit_loss_before_income_tax"]):
        return row["adjusted_profit_loss_before_income_tax"]

    # Only apply other corrections if profit > 0
    if profit <= 0:
        return profit

    # If the row's year is different from the correction year, no changes should be made
    if row["year"] != year:
        return profit
  
    # 2. Correct domestic profits
    if row["iso_parent"] == row["iso_partner"]:
        profit *= (1 - corrections["domestic"].get(year, {}).get(row["iso_parent"], 0))

    # For the rest of the corrections, iso_parent should not equal iso_partner
    elif row["iso_parent"] != row["iso_partner"]:
        # 3. Correct foreign profits in tax havens
        profit *= (1 - corrections["taxhavens"].get(year, {}).get(row["iso_partner"], 0))
        
        # 4. Correct foreign profits in country groups
        profit *= (1 - corrections["countrygroups"].get(year, {}).get(row["iso_partner"], 0))
        
        # 5. Correct foreign profits based on peculiarities in home jurisdiction
        profit *= (1 - corrections["foreign"].get(year, {}).get(row["iso_parent"], 0))

        # 6. Correct foreign profits based on US specific tax haven correction for 2018
        if (row["iso_parent"] == "USA") and (row["iso_partner"] in tax_havens) and (year == 2018):
            profit *= 0.61  # 1 - 0.39

    return profit

# Apply the function and create new column with corrected values
for year in range(first_year, first_year + n_years):
    mask = cbcr["year"] == year
    corrected_values = cbcr.loc[mask].apply(lambda row: correct_for_dividend_double_counting(row, year), axis=1)
    cbcr.loc[mask, "profit_loss_before_income_tax_corrected"] = corrected_values

Create logged variables

In [ ]:
# List of columns to transform
variables_to_log = ['unrelated_party_revenues', 'profit_loss_before_income_tax', 'profit_loss_before_income_tax_corrected',
                        'adjusted_profit_loss_before_income_tax', 'income_tax_paid_on_cash_basis',
                        'n_employees', 'tangible_assets_except_cash','stated_capital','total_revenues','related_party_revenues','holding_or_managing_ip']

# Generate log values where needed
for col_name in variables_to_log:
    new_col_name = f'ln_{col_name}'
    cbcr[new_col_name] = np.log(1 + cbcr[col_name])

c:\Users\AlisonSchultz\anaconda3\envs\sotj_profit_shifting_estimates\Lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\AlisonSchultz\anaconda3\envs\sotj_profit_shifting_estimates\Lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\AlisonSchultz\anaconda3\envs\sotj_profit_shifting_estimates\Lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\AlisonSchultz\anaconda3\envs\sotj_profit_shifting_estimates\Lib\site-packages\pandas\core\arraylike.py:396: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
c:\Users\AlisonSchultz\anaconda3\envs\sotj_profit_shifting_estimates\Lib\site-packages\pandas\core\arraylike.py:396: Runtime

#### 1.3 Calculate ETRs
- We calculate ETRs for two reasons: 
    1. We use them to double check whether our profit correction makes sense (it does if corrected ETRs of foreign and domestic firms are closer to each other than before the correction.)
    2. In our profit shifting estimates, we only consider profit shifting to jurisdictions with an ETR of below 15%. We therefore need to know the ETR by jurisdiction.
- We calculate ETRs based on the reported profits and based on the profits that have been corrected in 1.2 over a five years rolling window (starting two years before the current year and ending 2 years after the current year)

In [ ]:
def calculate_etr(df):
    """Calculate ETRs from CbCR data (based on subgroups that pay tax)"""
    d = df.loc[df["income_tax_paid_on_cash_basis"] >= 0].groupby("iso_partner").sum()
    d["etr"] = d["income_tax_paid_on_cash_basis"] / d["profit_loss_before_income_tax"]
    d["etr_corrected"] = d["income_tax_paid_on_cash_basis"] / d["profit_loss_before_income_tax_corrected"]
    return d[["etr", "etr_corrected"]]

def main_etrs(file, g="Sub-groups with positive profits"):
    """Calculate ETRs from CbCR data for each year using a rolling window."""
    df = file[["iso_parent", "iso_partner", "year", "partner_jurisdiction", "grouping", "income_tax_paid_on_cash_basis", "profit_loss_before_income_tax", "profit_loss_before_income_tax_corrected"]]
    df = df.loc[df["grouping"] == g]

    results = pd.DataFrame()

    # Get unique years
    unique_years = df["year"].unique()

    for year in unique_years:
        # Define the rolling window period
        start_year = year - 2
        end_year = year + 2

        # Filter data for the rolling window period
        df_window = df[(df["year"] >= start_year) & (df["year"] <= end_year)]
        df_window = df_window.dropna(subset=["income_tax_paid_on_cash_basis"])

        # Calculate ETRs for domestic, foreign, and average data within the rolling window
        df_foreign = df_window[df_window["iso_parent"] != df_window["iso_partner"]]
        df_domestic = df_window[df_window["iso_parent"] == df_window["iso_partner"]]
        df_average = df_window

        df_etr_domestic = calculate_etr(df_domestic).reset_index()
        df_etr_foreign = calculate_etr(df_foreign).reset_index()
        df_etr_average = calculate_etr(df_average).reset_index()

        # Merge the dataframes on the "iso_partner" column
        df_etr = df_etr_domestic.merge(df_etr_foreign, on="iso_partner", how="outer", suffixes=("_domestic", "_foreign"))
        df_etr = df_etr.merge(df_etr_average, on="iso_partner", how="outer", suffixes=("", "_average"))

        # Rename the columns for clarity
        df_etr.columns = ["iso_partner", "etr_domestic", "etr_domestic_corrected", "etr_foreign", "etr_foreign_corrected", "etr_average", "etr_average_corrected"]

        # Add a "Year" column to df_etr
        df_etr["year"] = year

        # Append the ETRs for this year to the results DataFrame
        results = pd.concat([results, df_etr], axis=0)

    return results


In [ ]:
# Calculate all relevant ETRs
etrs = main_etrs(cbcr)

# Adjust domestic ETRs of countries which do not report positive profits
# For those countries, we calculate ETRs based on all sub-groups, not only sub-groups with positive profits
reporting_countries_with_no_positive_profits = set(
    cbcr.loc[(cbcr["iso_parent"] == cbcr["iso_partner"]) & (cbcr["grouping"] == "Total (All sub-groups)"), "iso_parent"]
    ) - set(
    cbcr.loc[(cbcr["iso_parent"] == cbcr["iso_partner"]) & (cbcr["grouping"] == "Sub-groups with positive profits"), "iso_parent"]
    )
cbcr_countries_no_positive_profits = cbcr[cbcr["iso_parent"].isin(reporting_countries_with_no_positive_profits)]
cbcr_countries_no_positive_profits = cbcr_countries_no_positive_profits[cbcr_countries_no_positive_profits["iso_parent"] == cbcr_countries_no_positive_profits["iso_partner"]]
etrs_no_positive_profits = main_etrs(cbcr_countries_no_positive_profits, g="Total (All sub-groups)")
merged_df = etrs.merge(etrs_no_positive_profits[['iso_partner', 'year', 'etr_domestic', 'etr_domestic_corrected']], on=['iso_partner', 'year'], how='left', suffixes=('', '_update'))
# Update the values in 'etr_domestic' and 'etr_domestic_corrected' using the new values
merged_df['etr_domestic'] = merged_df['etr_domestic'].where(merged_df['etr_domestic_update'].isna(), merged_df['etr_domestic_update'])
merged_df['etr_domestic_corrected'] = merged_df['etr_domestic_corrected'].where(merged_df['etr_domestic_corrected_update'].isna(), merged_df['etr_domestic_corrected_update'])
etrs = merged_df.drop(columns=['etr_domestic_update', 'etr_domestic_corrected_update'])

In [ ]:
# merge ETRs to main dataset (MAYBE)
cbcr_etrs = cbcr.merge(etrs, on=["iso_partner", "year"], how="left")

### 2. Import other variables needed
- 2.1 Import corporate income tax rates
- 2.2 Import salary data
- 2.3 Import GDP and population data to impute missing values on salaries
- 2.4 Import health expenditure data for comparisons

In [ ]:
# Generate rows which should be present in each of the following datasets
sample_jur_year = []
for jur in partner_countries:
    for year in range(first_year, first_year + n_years):
        sample_jur_year.append((jur, year))

#### 2.1 Add corporate income tax rates
Before, the Table "" has to be downloaded from "" and stored in ""

In [ ]:
# Start with OECD data
cits_oecd_raw = pd.read_csv(cit_data_oecd)
columns_cit_data = ['REF_AREA', 'Measure','Targeting', 'TIME_PERIOD','OBS_VALUE']
cits_oecd_raw = cits_oecd_raw[columns_cit_data]
cits_oecd = cits_oecd_raw[
    (cits_oecd_raw['Measure'] == "Combined corporate income tax rate") & 
    (cits_oecd_raw['Targeting'] == "Statutory") &
    (cits_oecd_raw['TIME_PERIOD'] >= first_year) & 
    (cits_oecd_raw['TIME_PERIOD'] <= first_year + n_years)
].rename(columns={'REF_AREA': 'iso_partner', 
                  'TIME_PERIOD': 'year', 
                  'OBS_VALUE': 'cit'})\
  .drop(columns=['Measure', 'Targeting'])
cits_oecd['cit'] = cits_oecd['cit']/100

# Add countries from Tax Foundation data that do not have OECD data but do have Tax Foundation data
cits_wide_tf = pd.read_excel(cit_data_taxfoundation)
columns_cit_data_tf = ["iso_3"] + list(range(first_year, first_year + n_years))
cits_wide_tf = cits_wide_tf[columns_cit_data_tf]
value_vars = list(range(first_year, first_year + n_years))
cits_tf = cits_wide_tf.melt(id_vars=['iso_3'], 
                  value_vars=value_vars, 
                  var_name='Year', 
                  value_name='cit')
cits_tf.rename(columns={"iso_3": "iso_partner", "Year":"year"}, inplace=True)
cits_tf['cit'] = cits_tf['cit']/100

# Combine CITs from both sources
cits = pd.merge(cits_oecd, cits_tf, on=["iso_partner", "year"], how="outer", suffixes=('_cits', '_cits_tf'))
cits['cit'] = cits['cit_cits'].combine_first(cits['cit_cits_tf'])
cits = cits[['iso_partner', 'year', 'cit']].drop_duplicates().reset_index(drop=True)

In [ ]:
# Replace or add CIT rates for Martinique with the French and Bouvet Island with the Norwegian rate
for year in cits['year'].unique():
    # Replace or add CIT rate for Martinique (using France's CIT)
    if not ((cits['iso_partner'] == 'MTQ') & (cits['year'] == year)).any():
        new_row = pd.DataFrame({'iso_partner': ['MTQ'], 'year': [year], 
                                'cit': [cits.loc[(cits['iso_partner'] == 'FRA') & (cits['year'] == year), 'cit'].values[0]]})
        cits = pd.concat([cits, new_row], ignore_index=True)
    else:
        cits.loc[(cits['iso_partner'] == 'MTQ') & (cits['year'] == year), 'cit'] = cits.loc[(cits['iso_partner'] == 'FRA') & (cits['year'] == year), 'cit'].values[0]
    
    # Replace or add CIT rate for Bouvet Island (using Norway's CIT)
    if not ((cits['iso_partner'] == 'BVT') & (cits['year'] == year)).any():
        new_row = pd.DataFrame({'iso_partner': ['BVT'], 'year': [year], 
                                'cit': [cits.loc[(cits['iso_partner'] == 'NOR') & (cits['year'] == year), 'cit'].values[0]]})
        cits = pd.concat([cits, new_row], ignore_index=True)
    else:
        cits.loc[(cits['iso_partner'] == 'BVT') & (cits['year'] == year), 'cit'] = cits.loc[(cits['iso_partner'] == 'NOR') & (cits['year'] == year), 'cit'].values[0]

# Add further missing countries manually
missing_cits = []
for jur_year in sample_jur_year:
    if not ((cits['iso_partner'] == jur_year[0]) & (cits['year'] == jur_year[1])).any():
        missing_cits.append({'iso_partner': jur_year[0], 'year': jur_year[1]})
cits = pd.concat([cits, pd.DataFrame(missing_cits)], ignore_index=True)

# Replace missing CITs manually following Javier: TODO: SUBSTITUTE WITH STH MORE TRUSTWORTHY
cits.loc[cits['iso_partner'] == 'MLT', 'cit'] *= 1/7  # adjust Malta's cit by the 6/7th rule
cits.loc[cits['iso_partner'] == 'GIB', 'cit'] = 0     # adjust Gibraltar's CIT as it only applies to resident income
cits.loc[cits['iso_partner'] == 'MCO', 'cit'] = 0     # adjust MCO because ??
cits.loc[cits['iso_partner'] == 'AND', 'cit'] = 0     # adjust because ??
cits.loc[cits['iso_partner'] == 'CAF', 'cit'] = 0.3   # adjust because ??
cits.loc[cits['iso_partner'] == 'HTI', 'cit'] = 0.3   # adjust because ??
cits.loc[cits['iso_partner'] == 'YEM', 'cit'] = 0.2   # adjust because ??
cits.loc[cits['iso_partner'] == 'NCL', 'cit'] = 0     # Adjust to zero as only NCL income is taxable
cits.loc[cits['iso_partner'] == 'PRK', 'cit'] = 0.325 # 
cits.loc[cits['iso_partner'] == 'COD', 'cit'] = 0.28  # 
cits.loc[cits['iso_partner'] == 'TLS', 'cit'] = 0.10  # 
cits.loc[cits['iso_partner'] == 'USA', 'cit'] = 0.27  # Include state level taxes for US
cits.loc[cits['iso_partner'] == 'MHL', 'cit'] = 0     # https://www.consilium.europa.eu/en/press/press-releases/2023/02/14/taxation-british-virgin-islands-costa-rica-marshall-islands-and-russia-added-to-eu-list-of-non-cooperative-jurisdictions-for-tax-purposes/
cits.loc[cits['iso_partner'] == 'GLP', 'cit'] = .15   # https://www.confiduss.com/en/jurisdictions/guadeloupe/economy/
cits.loc[cits['iso_partner'] == 'GUF', 'cit'] = .28   # https://thetradecouncil.com/2021/07/04/corporate-income-tax-in-french-guiana/
cits.loc[cits['iso_partner'] == 'IOT', 'cit'] = np.nan
cits.loc[cits['iso_partner'] == 'PLW', 'cit'] = 0     # https://orbitax.com/taxhub/corporatetaxrates/PW/Palau
cits.loc[cits['iso_partner'] == 'PYF', 'cit'] = .27   # https://orbitax.com/taxhub/countrychapters/PF/French-Polynesia/7890123caa2f4bbc950c93677678bece/Corporate-Income-Tax-588
cits.loc[cits['iso_partner'] == 'REU', 'cit'] = .15   # https://www.confiduss.com/en/jurisdictions/reunion-island/
cits.loc[cits['iso_partner'] == 'SOM', 'cit'] = .3    # https://sominvest.gov.so/procedures/tax-regime/
cits.loc[cits['iso_partner'] == 'XKV', 'cit'] = .1    # https://taxsummaries.pwc.com/kosovo/corporate/taxes-on-corporate-income
cits.loc[cits['iso_partner'] == 'IOT', 'cit'] = 0     # zero taxes for British Indian Ocean territory
cits.loc[cits['iso_partner'] == 'SMR', 'cit'] = 0.17  # https://www.orbitax.com/taxhub/countrychapters/SM/San%20Marino/f422ca9b24bb422f820ed2741b8b2b00/CorporateProfit-Taxes-591


In [ ]:
# merge CITs to main dataset (MAYBE)
cbcr_etrs_cits = cbcr_etrs.merge(cits, on=["iso_partner", "year"], how="left")

#### 2.2 Add GDP and population
First download "" from "" and store it in ""

In [ ]:
gdp_population_long = pd.read_csv(gdp_population_data)
years = list(range(first_year, first_year + n_years))
formatted_years = [f"{year} [YR{year}]" for year in years]
columns_gdp_population_data = ["Series Name", "Country Code"] + formatted_years
gdp_population_long = gdp_population_long[columns_gdp_population_data]

gdp_population_long = gdp_population_long.melt(id_vars=["Country Code", "Series Name"], 
                    value_vars=[f"{year} [YR{year}]" for year in range(first_year, first_year + n_years)],
                    var_name="Year",
                    value_name="Value")
gdp_population = gdp_population_long.pivot_table(index=["Country Code", "Year"], 
                                 columns="Series Name", 
                                 values="Value", 
                                 aggfunc='first').reset_index() # aggfunc is not relevant here as there are no duplicates
gdp_population = gdp_population.rename(columns={"Country Code": "iso_partner","Year":"year","GDP (current US$)": "gdp_current_usd", "Population, total": "population"})
gdp_population['year'] = gdp_population['year'].str.extract(r'(\d{4})').astype(int)
gdp_population['gdp_current_usd'].replace('..', np.nan, inplace=True)
gdp_population['gdp_current_usd'] = pd.to_numeric(gdp_population['gdp_current_usd'], errors='coerce')
gdp_population['population'].replace('......', np.nan, inplace=True)
gdp_population['population'] = pd.to_numeric(gdp_population['population'], errors='coerce')

In [ ]:
# Add missing countries
missing_gdp_population = []
for jur_year in sample_jur_year:
    if not ((gdp_population['iso_partner'] == jur_year[0]) & (gdp_population['year'] == jur_year[1])).any():
        missing_gdp_population.append({'iso_partner': jur_year[0], 'year': jur_year[1]})
gdp_population = pd.concat([gdp_population, pd.DataFrame(missing_gdp_population)], ignore_index=True)

# Impute missing data from other sources, in particular GDP and population as this will be used to impute other values
# Taiwan
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 543.08 * 1e9 # https://www.statista.com/statistics/727589/gross-domestic-product-gdp-in-taiwan/
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 590.73 * 1e9 # https://www.statista.com/statistics/727589/gross-domestic-product-gdp-in-taiwan/
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 609.2 * 1e9  # https://www.statista.com/statistics/727589/gross-domestic-product-gdp-in-taiwan/
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 611.4  * 1e9 # https://www.statista.com/statistics/727589/gross-domestic-product-gdp-in-taiwan/
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2020), 'gdp_current_usd'] =  673.18 * 1e9 # https://www.statista.com/statistics/727589/gross-domestic-product-gdp-in-taiwan/
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 773.04 * 1e9

gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2016), 'population'] = 23512136 # 2015 as no 2016 data, https://worldpopulationreview.com/countries/taiwan-population
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2017), 'population'] = 23665024 #https://worldpopulationreview.com/countries/taiwan-population
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2018), 'population'] = 23726185 #https://worldpopulationreview.com/countries/taiwan-population
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2019), 'population'] = 23674138
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2020), 'population'] = 23663459
gdp_population.loc[(gdp_population['iso_partner'] == 'TWN') & (gdp_population['year'] == 2021), 'population'] = 23663459 #2020 data


# Anguilla
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 331 * 1e6    # 2015 value (as 2016 is not available): http://data.un.org/en/iso/ai.html
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & ((gdp_population['year'] == 2017) | (gdp_population['year'] == 2018)), 'gdp_current_usd'] = 322 * 1e6 # same for 2017 and 2018, as there is no 2017 and 2016 data: http://data.un.org/en/iso/ai.html
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 380 * 1e6 # 2021 data
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 380 * 1e6 # 2021 data
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 380 * 1e6 # 2021 data


gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2016), 'population'] = 14.3 * 1e3 # https://worldpopulationreview.com/countries/anguilla-population
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2017), 'population'] = 14.4 * 1e3  # https://worldpopulationreview.com/countries/anguilla-population
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2018), 'population'] = 14.7 * 1e3  # https://worldpopulationreview.com/countries/anguilla-population
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2019), 'population'] = 14.8 * 1e3 
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2020), 'population'] = 14.8 * 1e3 
gdp_population.loc[(gdp_population['iso_partner'] == 'AIA') & (gdp_population['year'] == 2021), 'population'] = 14.5 * 1e3 

# Cook Islands
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 287988 * 1e3 * 0.69 # https://stats.pacificdata.org
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 345587 * 1e3 * 0.7 # https://stats.pacificdata.org
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 391959 * 1e3 * 0.67 # https://stats.pacificdata.org
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 593585 * 1e3 * 0.64 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=26&tm=gdp&df[ds]=ds%3ASPC2&df[id]=DF_NATIONAL_ACCOUNTS&df[ag]=SPC&df[vs]=1.0&pd=2012%2C&dq=A.DOM..GDPC&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 397791 * 1e3 * 0.7 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=26&tm=gdp&df[ds]=ds%3ASPC2&df[id]=DF_NATIONAL_ACCOUNTS&df[ag]=SPC&df[vs]=1.0&pd=2012%2C&dq=A.DOM..GDPC&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 349192 * 1e3 * 0.71 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=26&tm=gdp&df[ds]=ds%3ASPC2&df[id]=DF_NATIONAL_ACCOUNTS&df[ag]=SPC&df[vs]=1.0&pd=2012%2C&dq=A.DOM..GDPC&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false

gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2016), 'population'] = 15076 # 2017, https://stats.pacificdata.org/vis?pg=0&bp=true&snb=50&tm=population&df[ds]=ds%3ASPC2&df[id]=DF_POP_PROJ&df[ag]=SPC&df[vs]=3.0&pd=2017%2C2027&dq=A..MIDYEARPOPEST._T._T&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2017), 'population'] = 15076 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=50&tm=population&df[ds]=ds%3ASPC2&df[id]=DF_POP_PROJ&df[ag]=SPC&df[vs]=3.0&pd=2017%2C2027&dq=A..MIDYEARPOPEST._T._T&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2018), 'population'] = 15153 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=50&tm=population&df[ds]=ds%3ASPC2&df[id]=DF_POP_PROJ&df[ag]=SPC&df[vs]=3.0&pd=2017%2C2027&dq=A..MIDYEARPOPEST._T._T&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2019), 'population'] = 15216 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=50&tm=population&df[ds]=ds%3ASPC2&df[id]=DF_POP_PROJ&df[ag]=SPC&df[vs]=3.0&pd=2017%2C2027&dq=A..MIDYEARPOPEST._T._T&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2020), 'population'] = 15281 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=50&tm=population&df[ds]=ds%3ASPC2&df[id]=DF_POP_PROJ&df[ag]=SPC&df[vs]=3.0&pd=2017%2C2027&dq=A..MIDYEARPOPEST._T._T&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false
gdp_population.loc[(gdp_population['iso_partner'] == 'COK') & (gdp_population['year'] == 2021), 'population'] = 15342 # https://stats.pacificdata.org/vis?pg=0&bp=true&snb=50&tm=population&df[ds]=ds%3ASPC2&df[id]=DF_POP_PROJ&df[ag]=SPC&df[vs]=3.0&pd=2017%2C2027&dq=A..MIDYEARPOPEST._T._T&ly[rw]=GEO_PICT&ly[cl]=TIME_PERIOD&to[TIME_PERIOD]=false

# Guernsey
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 2934 * 1.3552 * 1e6 # https://gov.gg/CHttpHandler.ashx?id=160890&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 3101 * 1.289 * 1e6  # https://gov.gg/CHttpHandler.ashx?id=160890&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 3170 * 1.3349 * 1e6 # https://gov.gg/CHttpHandler.ashx?id=160890&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 3248 * 1e6 * 1.31 # https://gov.gg/CHttpHandler.ashx?id=160890&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 3125 * 1e6 * 1.32 # https://gov.gg/CHttpHandler.ashx?id=160890&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 3446 * 1e6 * 1.34 # https://gov.gg/CHttpHandler.ashx?id=160890&p=0

gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2016), 'population'] = 61908 # https://gov.gg/CHttpHandler.ashx?id=121746&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2017), 'population'] = 62046 # https://gov.gg/CHttpHandler.ashx?id=121746&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2018), 'population'] = 62506 # https://gov.gg/CHttpHandler.ashx?id=121746&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2019), 'population'] = 62885 # https://www.gov.gg/CHttpHandler.ashx?id=169995&p=0
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2020), 'population'] = 63156
gdp_population.loc[(gdp_population['iso_partner'] == 'GGY') & (gdp_population['year'] == 2021), 'population'] = 63664


# Gibraltar
gdp_population.loc[(gdp_population['iso_partner'] == 'GIB') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 2.344 * 1.3349 * 1e9 # 2018 data, as no older data exists, https://en.wikipedia.org/wiki/Economy_of_Gibraltar
gdp_population.loc[(gdp_population['iso_partner'] == 'GIB') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 2.344 * 1.3349 * 1e9 # 2018 data, as no older data exists, https://en.wikipedia.org/wiki/Economy_of_Gibraltar
gdp_population.loc[(gdp_population['iso_partner'] == 'GIB') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 2.344 * 1.3349 * 1e9 # https://en.wikipedia.org/wiki/Economy_of_Gibraltar
gdp_population.loc[(gdp_population['iso_partner'] == 'GIB') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 2.344 * 1.3349 * 1e9
gdp_population.loc[(gdp_population['iso_partner'] == 'GIB') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 2.344 * 1.3349 * 1e9
gdp_population.loc[(gdp_population['iso_partner'] == 'GIB') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 2.344 * 1.3349 * 1e9

# Guadeloupe
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 8712.316/0.904 * 1e6    # https://www.ceicdata.com/en/france/esa-2010-gdp-by-region-current-prices-base-2014/gdp-guadeloupe
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 8803.461/0.8865 * 1e6 # https://www.ceicdata.com/en/france/esa-2010-gdp-by-region-current-prices-base-2014/gdp-guadeloupe
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 9025.467*1.1811 * 1e6 # https://www.ceicdata.com/en/france/esa-2010-gdp-by-region-current-prices-base-2014/gdp-guadeloupe
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 9268.066 * 1.11 * 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 8857.257 * 1.21 * 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 9169.070 * 1.13 * 1e6

gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2016), 'population'] = 395700 # Google
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2017), 'population'] = 402119 # https://en.wikipedia.org/wiki/Guadeloupe
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2018), 'population'] = 402119 # 2017 data as no 2018 data available: https://en.wikipedia.org/wiki/Guadeloupe
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2019), 'population'] = 384239 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2020), 'population'] = 383559 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls
gdp_population.loc[(gdp_population['iso_partner'] == 'GLP') & (gdp_population['year'] == 2021), 'population'] = 384315 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls


# French Guiana
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 4131/0.904 * 1e6    # https://www.insee.fr/en/statistiques/serie/010751772#Tableau
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 4127/0.8865 * 1e6   # https://www.insee.fr/en/statistiques/serie/010751772#Tableau
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 4353 * 1.1811 * 1e6 # https://www.insee.fr/en/statistiques/serie/010751772#Tableau
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 4431 * 1.11 * 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 4275 * 1.21 * 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 4450 * 1.13 * 1e6

gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2016), 'population'] = 267821 # https://statisticstimes.com/demographics/country/french-guiana-population.php
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2017), 'population'] = 275191 # https://statisticstimes.com/demographics/country/french-guiana-population.php
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2018), 'population'] = 282938 # https://statisticstimes.com/demographics/country/french-guiana-population.php
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2019), 'population'] = 281678 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2020), 'population'] = 285133 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls
gdp_population.loc[(gdp_population['iso_partner'] == 'GUF') & (gdp_population['year'] == 2021), 'population'] = 286618 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls


# British Indian Ocean Territory
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 1e6   
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 1e6

gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2016), 'population'] = 3000 # https://en.wikipedia.org/wiki/British_Indian_Ocean_Territory#Economy
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2017), 'population'] = 3000 # https://en.wikipedia.org/wiki/British_Indian_Ocean_Territory#Economy
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2018), 'population'] = 3000 # https://en.wikipedia.org/wiki/British_Indian_Ocean_Territory#Economy
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2019), 'population'] = 3000
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2020), 'population'] = 3000
gdp_population.loc[(gdp_population['iso_partner'] == 'IOT') & (gdp_population['year'] == 2021), 'population'] = 3000

# Jersey
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2016), 'gdp_current_usd'] =  4.11 * 1.3552 * 1e9  # https://www.gov.je/news/2017/pages/gvaandgdp2016.aspx
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2017), 'gdp_current_usd'] =  4.304 * 1.289 * 1e9 # https://www.gov.je/news/2018/pages/measuringjerseyseconomy2017.aspx
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2018), 'gdp_current_usd'] =  4.642 * 1.3349 * 1e9 # https://www.gov.je/news/2019/pages/measuringjerseyseconomy2018.aspx
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 4.885 * 1.31 * 1e9  # https://www.gov.je/news/2020/pages/measuringjerseyseconomy2019.aspx
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 4.528 * 1.32 * 1e9  # https://www.gov.je/SiteCollectionDocuments/Government%20and%20administration/R%20GVA%20and%20GDP%202020%2020211001%20SJ.pdf
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 5.087 * 1.34 * 1e9  # https://www.gov.je/SiteCollectionDocuments/Government%20and%20administration/R%20GVA%20and%20GDP%202021%2020221005%20SJ.pdf

gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2016), 'population'] = 102200
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2017), 'population'] = 102700
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2018), 'population'] = 103300
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2019), 'population'] = 103200 
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2020), 'population'] = 103300 
gdp_population.loc[(gdp_population['iso_partner'] == 'JEY') & (gdp_population['year'] == 2021), 'population'] = 103100

# Reunión
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 18065/0.904 * 1e6    # https://www.insee.fr/en/statistiques/serie/010751763
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 18555/0.8865 * 1e6   # https://www.insee.fr/en/statistiques/serie/010751763
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 18822 * 1.1811 * 1e6 # https://www.insee.fr/en/statistiques/serie/010751763
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 19367 * 1.11 * 1e6 # https://www.insee.fr/en/statistiques/serie/010751763
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 19032 * 1.21 * 1e6 # https://www.insee.fr/en/statistiques/serie/010751763
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 20412 * 1.13 * 1e6 # https://www.insee.fr/en/statistiques/serie/010751763

gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2016), 'population'] = 926628 # https://www.worldometers.info/world-population/reunion-population/
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2017), 'population'] = 932739 # https://www.worldometers.info/world-population/reunion-population/
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2018), 'population'] = 941187 # https://www.worldometers.info/world-population/reunion-population/
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2019), 'population'] = 861210 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2020), 'population'] = 863083 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls
gdp_population.loc[(gdp_population['iso_partner'] == 'REU') & (gdp_population['year'] == 2021), 'population'] = 871157 # https://www.insee.fr/fr/statistiques/fichier/7752095/estim-pop-nreg-sexe-gca-1975-2024.xls

# Wallis and Futuna
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 139500 * 1e3 # https://stats.pacificdata.org/
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 139500 * 1e3 # https://stats.pacificdata.org/
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 139500 * 1e3 # https://stats.pacificdata.org/
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 139500 * 1e3 
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 139500 * 1e3 
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 139500 * 1e3 

gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2016), 'population'] = 12060 # https://www.worldometers.info/world-population/wallis-and-futuna-islands-population/
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2017), 'population'] = 11936 # https://www.worldometers.info/world-population/wallis-and-futuna-islands-population/
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2018), 'population'] = 11816 # https://www.worldometers.info/world-population/wallis-and-futuna-islands-population/
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2019), 'population'] = 11502
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2020), 'population'] = 11441
gdp_population.loc[(gdp_population['iso_partner'] == 'WLF') & (gdp_population['year'] == 2021), 'population'] = 11369

# Kosovo
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 6.68 * 1e9 # https://de.statista.com/statistik/daten/studie/415738/umfrage/bruttoinlandsprodukt-bip-des-kosovo/
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 7.18 * 1e9 # https://de.statista.com/statistik/daten/studie/415738/umfrage/bruttoinlandsprodukt-bip-des-kosovo/
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 7.88 * 1e9 # https://de.statista.com/statistik/daten/studie/415738/umfrage/bruttoinlandsprodukt-bip-des-kosovo/
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 7.9 * 1e9
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 7.73 * 1e9
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 9.42 * 1e9

gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2016), 'population'] = 1777557 # https://data.worldbank.org/indicator/SP.POP.TOTL?locations=XK
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2017), 'population'] = 1791003 # https://data.worldbank.org/indicator/SP.POP.TOTL?locations=XK
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2018), 'population'] = 1797085 # https://data.worldbank.org/indicator/SP.POP.TOTL?locations=XK
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2019), 'population'] = 1788878
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2020), 'population'] = 1790133
gdp_population.loc[(gdp_population['iso_partner'] == 'XKV') & (gdp_population['year'] == 2021), 'population'] = 1786038

# Saint Martin
gdp_population.loc[gdp_population['iso_partner'] == 'MAF', 'gdp'] = 772921776 # 2014 data as other years are not available, https://data.worldbank.org/indicator/NY.GDP.MKTP.CD?locations=MF
# North Korea
gdp_population.loc[gdp_population['iso_partner'] == 'PRK', 'gdp'] = 772921776 # 2015 data as no other years available, https://www.cia.gov/the-world-factbook/countries/korea-north/#economy
# South Sudan
gdp_population.loc[(gdp_population['iso_partner'] == 'SSD') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 2.9 * 1e9  # Google
gdp_population.loc[(gdp_population['iso_partner'] == 'SSD') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 1.8 * 1e9  # Google
gdp_population.loc[(gdp_population['iso_partner'] == 'SSD') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 3.12 * 1e9 # https://www.statista.com/statistics/727342/gross-domestic-product-gdp-in-south-sudan/
gdp_population.loc[(gdp_population['iso_partner'] == 'SSD') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 4.04 * 1e9  
gdp_population.loc[(gdp_population['iso_partner'] == 'SSD') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 5.42 * 1e9 
gdp_population.loc[(gdp_population['iso_partner'] == 'SSD') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 5.94 * 1e9 

# Venezuela
gdp_population.loc[(gdp_population['iso_partner'] == 'VEN') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 112.92 * 1e9 # https://www.statista.com/statistics/370937/gross-domestic-product-gdp-in-venezuela/
gdp_population.loc[(gdp_population['iso_partner'] == 'VEN') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 115.88 * 1e9 # https://www.statista.com/statistics/370937/gross-domestic-product-gdp-in-venezuela/
gdp_population.loc[(gdp_population['iso_partner'] == 'VEN') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 102.02 * 1e9 # https://www.statista.com/statistics/370937/gross-domestic-product-gdp-in-venezuela/
gdp_population.loc[(gdp_population['iso_partner'] == 'VEN') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 73 * 1e9
gdp_population.loc[(gdp_population['iso_partner'] == 'VEN') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 43.79 * 1e9
gdp_population.loc[(gdp_population['iso_partner'] == 'VEN') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 57.67 * 1e9

# British Virgin Islands
gdp_population.loc[(gdp_population['iso_partner'] == 'VGB') & (gdp_population['year'] == 2016), 'gdp_current_usd'] = 1279 * 1e6 # 2015 values as no 2016 values available, https://unctadstat.unctad.org/countryprofile/generalprofile/en-gb/092/index.html
gdp_population.loc[(gdp_population['iso_partner'] == 'VGB') & (gdp_population['year'] == 2017), 'gdp_current_usd'] = 1279 * 1e6 # 2015 values as no 2016 values available, https://unctadstat.unctad.org/countryprofile/generalprofile/en-gb/092/index.html
gdp_population.loc[(gdp_population['iso_partner'] == 'VGB') & (gdp_population['year'] == 2018), 'gdp_current_usd'] = 1653 * 1e6 # 2021 values as no 2018 values available, https://unctadstat.unctad.org/countryprofile/generalprofile/en-gb/092/index.html
gdp_population.loc[(gdp_population['iso_partner'] == 'VGB') & (gdp_population['year'] == 2019), 'gdp_current_usd'] = 1653 * 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'VGB') & (gdp_population['year'] == 2020), 'gdp_current_usd'] = 1653 * 1e6
gdp_population.loc[(gdp_population['iso_partner'] == 'VGB') & (gdp_population['year'] == 2021), 'gdp_current_usd'] = 1653 * 1e6

# if there's information on some years: replace missing values with average value over years
avg_gdp = gdp_population.groupby('iso_partner')['gdp_current_usd'].transform('mean')
gdp_population['gdp_current_usd'].fillna(avg_gdp, inplace=True)
avg_pop = gdp_population.groupby('iso_partner')['population'].transform('mean')
gdp_population['population'].fillna(avg_pop, inplace=True)

#### 2.3 Add salary data
First download "" from "" and store it in ""

In [ ]:
wages_fulltable = pd.read_csv(wage_data)
# Select relevant rows
both_sexes = wages_fulltable['sex'] == 'SEX_T'
all_occupations = wages_fulltable['classif1'] == 'OCU_SKILL_TOTAL'
in_usd = wages_fulltable['classif2'] == 'CUR_TYPE_USD'
relevant_years = (wages_fulltable['time'] >= first_year) & (wages_fulltable['time'] <= first_year + n_years)
relevant_rows = both_sexes & all_occupations & in_usd & relevant_years
wages = wages_fulltable.loc[relevant_rows, ['ref_area', 'time', 'obs_value']].reset_index()
wages.rename(columns={"ref_area": "iso_partner", "time": "year", "obs_value": "wage_monthly"}, inplace=True)

# Add missing countries
missing_wages = []
for jur_year in sample_jur_year:
    if not ((wages['iso_partner'] == jur_year[0]) & (wages['year'] == jur_year[1])).any():
        missing_wages.append({'iso_partner': jur_year[0], 'year': jur_year[1]})
wages = pd.concat([wages, pd.DataFrame(missing_wages)], ignore_index=True)

# if there's information on some years: replace missing values with average value over years
avg_wage = wages.groupby('iso_partner')['wage_monthly'].transform('mean')
wages['wage_monthly'].fillna(avg_wage, inplace=True)

# Include missing wage data consistent with the SOTJ 2023, some of these wages have been transformed to US dollars based on the applicable average exchange rate in a given year
wages.loc[wages['iso_partner'] == 'AIA', 'wage_monthly'] = 74620.18 / 12  # https://www.salaryexpert.com/salary/area/netherlands/the-valley--anguilla 
wages.loc[wages['iso_partner'] == 'COK', 'wage_monthly'] = 2913.46        # https://worldsalaries.com/average-salary-in-cook-islands/
wages.loc[wages['iso_partner'] == 'GGY', 'wage_monthly'] = 9859.02        # https://www.salaryexplorer.com/average-salary-wage-comparison-guernsey-c90
wages.loc[wages['iso_partner'] == 'GIB', 'wage_monthly'] = 4462           # https://worldsalaries.com/average-salary-in-gibraltar/
wages.loc[wages['iso_partner'] == 'GLP', 'wage_monthly']= 2277.85         # https://www.salaryexplorer.com/average-salary-wage-comparison-guadalupe-t1601 
wages.loc[wages['iso_partner'] == 'GUF', 'wage_monthly'] = 16983.03/12    # https://www.insee.fr/en/statistiques/serie/001781953
wages.loc[wages['iso_partner'] == 'JEY', 'wage_monthly']= 1098.36*4       # https://www.gov.je/StatisticsPerformance/EmploymentEarnings/pages/earningsincomestatistics.aspx
wages.loc[wages['iso_partner'] == 'MAF', 'wage_monthly'] = 46000/12       # https://www.sint-maarten.net/population/life
wages.loc[wages['iso_partner'] == 'TWN', 'wage_monthly']= 21689/12        # https://nhglobalpartners.com/countries/taiwan/hiring-employees/average-salary/
wages.loc[wages['iso_partner'] == 'WLF', 'wage_monthly'] = 625.75         # https://www.salaryexplorer.com/average-salary-wage-comparison-wallis-and-futuna-c239

In [ ]:
# merge wages to main dataset (MAYBE)
gdp_population_wages = gdp_population.merge(wages, on=["iso_partner", "year"], how="left")

#### 2.3.2 Impute wages with OLS estimate based on log(gdp) and log(population)

In [ ]:
# Identify implausible values to leave them out of the regression
offsample = []  # Create list of observations to be removed from the regression
for i, x, y in zip(
    gdp_population_wages["iso_partner"],
    gdp_population_wages["gdp_current_usd"] / gdp_population_wages["population"],
    gdp_population_wages["wage_monthly"],
):  # Detect implausible values for monthly wages
    if (np.log(x / 12 / y) > np.log2(1.5)) or (
        np.log(x / 12 / y) < -2
    ):  # rule of thumb which values are plausible (and which not)
        offsample.append(i)
offsample = list(dict.fromkeys(offsample))
# remove implausible values and substitue missing values
ols_sample = gdp_population_wages[~gdp_population_wages["iso_partner"].isin(offsample)]
ols_regression = smf.ols(
    formula = "np.log(wage_monthly) ~ np.log(gdp_current_usd) + np.log(population)", data = ols_sample
)
# Predict wage based on GDP and population to substitute missing or implausible values
ols_fitted_values = ols_regression.fit()
# Include predicted wages in dataset
gdp_population_wages["pred_wage_monthly"] = np.exp(
    ols_fitted_values.predict(gdp_population_wages)
) 
# substitute wages with predicted wages if wages are missing
gdp_population_wages.loc[np.isnan(gdp_population_wages["wage_monthly"]), "wage_monthly"] = gdp_population_wages.loc[
    np.isnan(gdp_population_wages["wage_monthly"]), "pred_wage_monthly"
] 
del gdp_population_wages["pred_wage_monthly"]  # delete predicted wages

Log values

In [ ]:
# List of columns to transform
variables_to_log = ['wage_monthly','gdp_current_usd','population']

# Generate log values where needed
for col_name in variables_to_log:
    new_col_name = f'ln_{col_name}'
    gdp_population_wages[new_col_name] = np.log(1 + gdp_population_wages[col_name])

In [ ]:
# merge GDP, population, and wage data to main dataset (MAYBE)
cbcr_etrs_cits_gdp_wages = cbcr_etrs_cits.merge(gdp_population_wages, on=["iso_partner", "year"], how="left")

#### 2.4 Add Health data
First download "" from "" and store it in ""

In [ ]:
health_expenditure_wide = pd.read_excel(health_expenditure_data)
health_expenditure_wide = health_expenditure_wide[health_expenditure_wide['Indicators'] == "Domestic General Government Health Expenditure (GGHE-D)"]
health_expenditure_wide = health_expenditure_wide.dropna(subset=['Countries'])
health_expenditure_wide["iso_partner"] = health_expenditure_wide["Countries"].map(tjn_tools.get_iso3)
health_expenditure_wide["iso_partner"].loc[health_expenditure_wide["Countries"] == "Netherlands (Kingdom of the)"] = "NLD"
print("Netherlands value corrected")
health_expenditure_wide["iso_partner"].loc[health_expenditure_wide["Countries"] == "Türkiye"] = "TUR"
print("Turkey value corrected")
columns_health_data = ["iso_partner"] + [str(year) for year in range(first_year, first_year + n_years)]
health_expenditure_wide = health_expenditure_wide[columns_health_data]
value_vars = [str(year) for year in range(first_year, first_year + n_years)]
health_expenditure = health_expenditure_wide.melt(id_vars=['iso_partner'], 
                  value_vars=value_vars, 
                  var_name='year', 
                  value_name='gvt_health_expenditure')
health_expenditure['gvt_health_expenditure'] = health_expenditure['gvt_health_expenditure'] * 1e6 #Transform million USD value into USD value

Netherlands (Kingdom of the) not matched to any file
Türkiye not matched to any file
Netherlands value corrected
Turkey value corrected


Log values

In [ ]:
health_expenditure['gvt_health_expenditure'] = pd.to_numeric(health_expenditure['gvt_health_expenditure'], errors='coerce')
health_expenditure['ln_gvt_health_expenditure'] = np.log(1 + health_expenditure['gvt_health_expenditure'])

In [ ]:
# merge health expenditure data to main dataset (MAYBE)
health_expenditure['year'] = health_expenditure['year'].astype(int)
cbcr_etrs_cits_gdp_wages_health = cbcr_etrs_cits_gdp_wages.merge(health_expenditure, on=["iso_partner", "year"], how="left")

### 2.5 Add data on tax revenue
https://api.worldbank.org/v2/en/indicator/GC.TAX.TOTL.GD.ZS?downloadformat=csv

In [ ]:
tax_revenue_wide = pd.read_csv(tax_revenue_data, skiprows=4)
value_vars = [str(year) for year in range(first_year, first_year + n_years)]
tax_revenue = tax_revenue_wide.melt(id_vars=['Country Code'], 
                  value_vars=value_vars, 
                  var_name='year', 
                  value_name='tax_revenue_pct_gdp')
tax_revenue.rename(columns={"Country Code": "iso_partner"}, inplace=True)

In [ ]:
tax_revenue['year'] = tax_revenue['year'].astype(int)
cbcr_etrs_cits_gdp_wages_health_taxes = cbcr_etrs_cits_gdp_wages_health.merge(tax_revenue, on=["iso_partner","year"], how="left")
cbcr_etrs_cits_gdp_wages_health_taxes['tax_revenue_pct_gdp'] = pd.to_numeric(cbcr_etrs_cits_gdp_wages_health_taxes['tax_revenue_pct_gdp'], errors='coerce')
cbcr_etrs_cits_gdp_wages_health_taxes['gdp_current_usd'] = pd.to_numeric(cbcr_etrs_cits_gdp_wages_health_taxes['gdp_current_usd'], errors='coerce')
cbcr_etrs_cits_gdp_wages_health_taxes['tax_revenue_current_usd'] = cbcr_etrs_cits_gdp_wages_health_taxes['tax_revenue_pct_gdp']/100 * cbcr_etrs_cits_gdp_wages_health_taxes['gdp_current_usd']

#### 2.6 Add regions and country groups

In [ ]:
regions = pd.read_csv(unilateral_cross_data)
regions = pd.read_csv(unilateral_cross_data, usecols=['iso3', 'region_tjn','ukt','oecd_oct','oecd',]).dropna(subset=["iso3", 'region_tjn'])
regions.rename(columns={'iso3': 'iso_partner'}, inplace=True)

### Bring everything together

In [ ]:
cbcr_main_no_imputation = cbcr_etrs_cits_gdp_wages_health_taxes.merge(regions, on=["iso_partner"], how="left")
cbcr_main_no_imputation_allsubgroupsonly = cbcr_main_no_imputation[cbcr_main_no_imputation["grouping"] == "Total (All sub-groups)"]

cbcr_main_no_imputation.to_csv(f'{data_intermediate}/cbcr_main_no_imputation.csv')
cbcr_main_no_imputation_allsubgroupsonly.to_csv(f'{data_intermediate}/cbcr_main_no_imputation_allsubgroupsonly.csv')

#### 2.7 Double check dataset
No missings should be on wages, GDP, Population, CIT (exception for CIT if no rate is found which is the case for 'IOT' (British Indian Ocean Territory))

In [ ]:
cbcr_main_no_imputation = cbcr_main_no_imputation.sort_values(by=['iso_partner', 'year', 'n_employees', 'profit_loss_before_income_tax_corrected'], ascending=[True, True, False, False])
cbcr_check = cbcr_main_no_imputation.drop_duplicates(subset=['iso_partner', 'year'], keep='first')
cbcr_check = cbcr_check[~cbcr_check['iso_partner'].isin(country_groups)]
variables = ['profit_loss_before_income_tax_corrected', 'n_employees', 'wage_monthly', 'gdp_current_usd', 'population', 'gvt_health_expenditure','cit','etr_foreign_corrected']

for var in variables:
    missing_ids = cbcr_check[cbcr_check[var].isna()][['iso_partner', 'year']].values.tolist()
    if missing_ids:
        print(f"For variable {var}, IDs with missing values are: {missing_ids}")

For variable profit_loss_before_income_tax_corrected, IDs with missing values are: [['SMR', 2020], ['W', 2021]]
For variable n_employees, IDs with missing values are: [['SMR', 2020], ['W', 2021]]
For variable wage_monthly, IDs with missing values are: [['A', 2016], ['A', 2017], ['A', 2018], ['A', 2019], ['A', 2020], ['A', 2021], ['A_O', 2016], ['A_O', 2017], ['A_O', 2018], ['A_O', 2019], ['A_O', 2020], ['A_O', 2021], ['E', 2016], ['E', 2017], ['E', 2018], ['E', 2019], ['E', 2020], ['E', 2021], ['E_O', 2016], ['E_O', 2017], ['E_O', 2018], ['E_O', 2019], ['E_O', 2020], ['E_O', 2021], ['F', 2016], ['F', 2017], ['F', 2018], ['F', 2019], ['F', 2020], ['F', 2021], ['FLK', 2021], ['F_O', 2016], ['F_O', 2017], ['F_O', 2018], ['F_O', 2019], ['F_O', 2020], ['F_O', 2021], ['MTQ', 2019], ['MTQ', 2020], ['PRK', 2017], ['PRK', 2018], ['PRK', 2021], ['S', 2016], ['S', 2017], ['S', 2018], ['S', 2019], ['S', 2020], ['S', 2021], ['S_O', 2016], ['S_O', 2017], ['S_O', 2018], ['S_O', 2019], ['S_O', 2020], 

### 3.1 Create dataset for imputation

#### 3.1.1 Import Trade and Investment Related bilateral variables

In [ ]:
trade = pd.read_stata(comtrade_data)[['r_iso3','p_iso3','year','exports_tot','import_tot']]

In [ ]:
fdi = pd.read_stata(cdis_data)[['r_iso3','p_iso3','year','fdi_inward','fdi_outward']]

In [ ]:
pi = pd.read_stata(cpis_data)[['r_iso3','p_iso3','year','pi_inward','pi_outward']]

In [ ]:
bank_deposits = pd.read_csv(bis_data)[['r_iso3','p_iso3','year','claims','dclaims','liabilities','dliabilities']]
bank_deposits['claims'] = bank_deposits['claims'].fillna(bank_deposits['dclaims'])
bank_deposits['liabilities'] = bank_deposits['liabilities'].fillna(bank_deposits['dliabilities'])
bank_deposits = bank_deposits[['r_iso3','p_iso3','year','claims','liabilities']]

#### 3.1.2 Import gravity variables

In [ ]:
gravity = pd.read_csv(gravity_data)[['year','iso3_o','iso3_d','distw_arithmetic','transition_legalchange',
                                     'col45','col_dep_ever','comcol','comlang_ethno','comlang_off',
                                     'comleg_posttrans','comleg_pretrans','comrelig','contig',
                                     'sibling','fta_wto','heg_o','heg_d',
                                     'entry_proc_o','entry_proc_d','entry_time_o','entry_time_d','entry_tp_o',
                                     'entry_tp_d', 'entry_cost_o', 'entry_cost_d','gatt_o','gatt_d','eu_o','eu_d']]

C:\Users\AlisonSchultz\AppData\Local\Temp\ipykernel_28428\10017178.py:1: DtypeWarning: Columns (40) have mixed types. Specify dtype option on import or set low_memory=False.
  gravity = pd.read_csv(gravity_data)[['year','iso3_o','iso3_d','distw_arithmetic','transition_legalchange',


In [ ]:
# Update old ISO codes
gravity["iso3_o"] = gravity["iso3_o"].str.replace("TLS", "TMP")
gravity["iso3_d"] = gravity["iso3_d"].str.replace("TLS", "TMP")
gravity["iso3_o"] = gravity["iso3_o"].str.replace("ANT","CUW")
gravity["iso3_d"] = gravity["iso3_d"].str.replace("ANT","CUW")

gravity = gravity.rename(columns=lambda col: 'r_' + col[:-2] if col.endswith('_o') else ('p_' + col[:-2] if col.endswith('_d') else col))


In [67]:
def copy_data_from_other_country(distance_data, countries_to_impute_data, country_to_copy_from):
    """This function imputes data of another country ('country_to_copy_from') to the countries specified in 'countries_to_impute_data'.
       This can make sense for distance based variables of countries that are close to each other."""
    for country in countries_to_impute_data:
        x = distance_data.loc[distance_data["iso_parent"] == country_to_copy_from] # take rows from country to copy from (start when countries appear as origin country)
        x["iso_parent"] = country                                                  # change the country iso 3 to the one of the country to impute data
        distance_data = pd.concat([distance_data, x])                          # add the country with imputed data to main dataset                             
        x = distance_data.loc[distance_data["iso_partner"] == country_to_copy_from] # same operation for countries appearing as destination country: take rows from country to copy from
        x["iso_partner"] = country                                                  # change the country iso 3 to the one of the country to impute data
        distance_data = pd.concat([distance_data, x])                          # add the country with imputed data to main dataset    
    return distance_data

# Substitute values of countries that do not have values with countries closeby
gravity = copy_data_from_other_country(gravity,["JEY", "IMN", "GGY"],"GBR")
gravity = copy_data_from_other_country(gravity,["SRB","MNE"],"ALB")
gravity = copy_data_from_other_country(gravity,["MCO"],"AND")
gravity = copy_data_from_other_country(gravity,["SSD"],"SDN")
gravity = copy_data_from_other_country(gravity,["COD"],"COG")

C:\Users\AlisonSchultz\AppData\Local\Temp\ipykernel_28900\3087112486.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x["iso_parent"] = country                                                  # change the country iso 3 to the one of the country to impute data
C:\Users\AlisonSchultz\AppData\Local\Temp\ipykernel_28900\3087112486.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x["iso_partner"] = country                                                  # change the country iso 3 to the one of the countr

### 3.1.3 

#### 3.1.3 Import Orbis data about number of firms, employees and turnover in a country

In [70]:
orbis = pd.read_excel(orbis_data, na_values=["n.a."], keep_default_na=False)  # import Orbis GUOs
orbis = orbis.dropna(subset=["GUO - BvD ID number"])  # drop observations that do not have a BvD ID
orbis = orbis.loc[orbis["Country ISO code"] == orbis["GUO - BvD ID number"].str[:2]]  # keep only rows where BvD ID starts with ISO Code of country (row of home country)
orbis["max_turnover"] = orbis.groupby(["GUO - BvD ID number"])["Operating revenue (Turnover)\nth USD Last avail. yr"].transform(max)  # calculate highest turnover per GUO to delete duplicates later
orbis = orbis.loc[orbis["Operating revenue (Turnover)\nth USD Last avail. yr"] == orbis["max_turnover"]]  # only keep GUO with highest turnover
orbis = orbis.drop_duplicates(subset=["GUO - BvD ID number"])  # if highest turnover is duplicate, drop
orbis.loc[
    (orbis["Country ISO code"] != orbis["GUO - BvD ID number"].str[:2])
    & (orbis["GUO - BvD ID number"].str[:2] != "WW"),
    "Country ISO code",
] = orbis.loc[
    (orbis["Country ISO code"] != orbis["GUO - BvD ID number"].str[:2])
    & (orbis["GUO - BvD ID number"].str[:2] != "WW"),
    "GUO - BvD ID number",
].str[
    :2
]
orbis = (orbis.groupby("Country ISO code").agg({"GUO - BvD ID number": len,
    "Operating revenue (Turnover)\nth USD Last avail. yr": np.sum,
    "Number of employees\nLast avail. yr": np.sum,}).reset_index())
orbis.columns = [
    "iso_partner",
    "n_companies_orbis",
    "turnover_orbis",
    "n_employees_orbis",]  # Rename columns to produce understandable table
orbis["turnover_orbis"] *= 1000  # report turnover in USD
orbis["iso_partner"] = orbis["iso_partner"].map(tjn_tools.get_iso3)  # replace country by ISO3
orbis = orbis.dropna(subset=["iso_partner"])  # drop if no country info is provided, ie if no ISO3 was assigned

 not matched to any file
II not matched to any file


#### 3.1.4 Merge samples for imputation

In [ ]:
relevant_partner_countries = [country for country in partner_countries if country not in aggregated_country_groups]
cbcr_for_merge = cbcr_main_no_imputation_allsubgroupsonly[['iso_parent', 'iso_partner', 'year',
    'unrelated_party_revenues', 'income_tax_paid_on_cash_basis', 'n_employees', 'tangible_assets_except_cash', 'stated_capital', 'total_revenues', 'related_party_revenues',
    'holding_or_managing_ip', 'n_cbcr', 'n_cbcr_groups', 'n_entities', 'profit_loss_before_income_tax_corrected',
    'ln_unrelated_party_revenues', 'ln_profit_loss_before_income_tax_corrected',
    'ln_income_tax_paid_on_cash_basis', 'ln_n_employees',
    'ln_tangible_assets_except_cash', 'ln_stated_capital', 'ln_total_revenues', 'ln_related_party_revenues',
    'ln_holding_or_managing_ip', 'etr_domestic_corrected', 'etr_foreign_corrected',
    'etr_average_corrected', 'cit', 'gdp_current_usd', 'population', 'gdp', 'wage_monthly',
    'ln_wage_monthly', 'ln_gdp_current_usd', 'ln_population', 'gvt_health_expenditure', 'ln_gvt_health_expenditure',
    'region_tjn', 'g7', 'eu27', 'g20', 'ukt']]
all_bilateral_combinations = list(product(parent_countries, relevant_partner_countries))
all_bilateral_combinations = list(product(all_bilateral_combinations, range(first_year, first_year + n_years)))
imputation_sample = pd.DataFrame([(country_pair[0], country_pair[1], year) for country_pair, year in all_bilateral_combinations],columns=["iso_parent", "iso_partner", "year"])
imputation_sample = imputation_sample.merge(cbcr_for_merge, on=["iso_parent", "iso_partner", "year"], how='left')
imputation_sample = imputation_sample.merge(gravity, on=["iso_parent", "iso_partner"], how='left')
imputation_sample = imputation_sample.merge(orbis, on=["iso_partner"], how='left')
imputation_sample.replace([np.inf, -np.inf], np.nan, inplace=True) 


In [ ]:
# List of columns to log
variables_to_log = ["exports_tot",'imports_tot'
        "claims","liabilities",
        "pi_inward", "pi_outward",
        "fdi_inward", "fdi_outward",
        'n_companies_orbis',
        'turnover_orbis',
        'n_employees_orbis']

# Generate log values where needed
for col_name in variables_to_log:
    new_col_name = f'ln_{col_name}'
    imputation_sample[new_col_name] = np.log(1 + imputation_sample[col_name])

In [72]:
imputation_sample.to_csv(f'{data_intermediate}/imputation_sample.csv')

KeyError: "['g7', 'g20'] not in index"